In [1]:
import gmsh
import sys
from numpy import array

# Initialize Gmsh
gmsh.initialize()

# Set the Gmsh model name
gmsh.model.add("cube_surface")

mesh_size = 0.1

# Define the corner points of the cube
gmsh.model.geo.addPoint(0, 0, 0, mesh_size, 1)
gmsh.model.geo.addPoint(1, 0, 0, mesh_size, 2)
gmsh.model.geo.addPoint(1, 1, 0, mesh_size, 3)
gmsh.model.geo.addPoint(0, 1, 0, mesh_size, 4)
gmsh.model.geo.addPoint(0, 0, 1, mesh_size, 5)
gmsh.model.geo.addPoint(1, 0, 1, mesh_size, 6)
gmsh.model.geo.addPoint(1, 1, 1, mesh_size, 7)
gmsh.model.geo.addPoint(0, 1, 1, mesh_size, 8)

# Define lines connecting the points to create cube edges
gmsh.model.geo.addLine(1, 2, 1)
gmsh.model.geo.addLine(2, 3, 2)
gmsh.model.geo.addLine(3, 4, 3)
gmsh.model.geo.addLine(4, 1, 4)
gmsh.model.geo.addLine(5, 6, 5)
gmsh.model.geo.addLine(6, 7, 6)
gmsh.model.geo.addLine(7, 8, 7)
gmsh.model.geo.addLine(8, 5, 8)
gmsh.model.geo.addLine(1, 5, 9)
gmsh.model.geo.addLine(2, 6, 10)
gmsh.model.geo.addLine(3, 7, 11)
gmsh.model.geo.addLine(4, 8, 12)

# Define surfaces (the cube faces)
gmsh.model.geo.addCurveLoop([-4, -3, -2, -1], 1)
gmsh.model.geo.addCurveLoop([5, 6, 7, 8], 2)
gmsh.model.geo.addCurveLoop([1, 10, -5, -9], 3)
gmsh.model.geo.addCurveLoop([2, 11, -6, -10], 4)
gmsh.model.geo.addCurveLoop([3, 12, -7, -11], 5)
gmsh.model.geo.addCurveLoop([4, 9, -8, -12], 6)

# Add plane surfaces from the curve loops
gmsh.model.geo.addPlaneSurface([1], 1)
gmsh.model.geo.addPlaneSurface([2], 2)
gmsh.model.geo.addPlaneSurface([3], 3)
gmsh.model.geo.addPlaneSurface([4], 4)
gmsh.model.geo.addPlaneSurface([5], 5)
gmsh.model.geo.addPlaneSurface([6], 6)

# Synchronize the model to generate the surfaces in Gmsh
gmsh.model.geo.synchronize()

# Mesh the surfaces with triangles of order 2 (second order elements)
gmsh.model.mesh.setOrder(2)

# Ensure high-order element generation
gmsh.option.setNumber("Mesh.ElementOrder", 2)
gmsh.option.setNumber("Mesh.Algorithm", 1)  # Control the algorithm used for meshing

# Create a 2D mesh on the cube surfaces
gmsh.model.mesh.generate(2)

# Optionally, write the mesh to a file (e.g., in .msh format)
gmsh.write("refined_cube_surface_mesh.msh")

# Finalize the Gmsh session
gmsh.finalize()


In [2]:
import meshio
import numpy as np
from scipy.io import savemat

# Load the .msh file
mesh = meshio.read("refined_cube_surface_mesh.msh")

# Extract the 2D triangular elements (for second-order triangles, each triangle has 6 nodes)
triangle_tags = np.array(mesh.cells_dict['triangle6'])

# Build the list of nodes
Points = np.array(mesh.points)

# Build the connectivity matrix: each row represents a triangle, each column a node of the triangle  (+1 because MATLAB indexes starts from 1 and not 0)
ConnectivityList = np.array(triangle_tags + 1,dtype=np.float64)

# Build the triangle coordinates matrix
triangles_coords = np.zeros((triangle_tags.shape[0], 6, 3))
for i, triangle in enumerate(triangle_tags):
    for j, node_index in enumerate(triangle):
        triangles_coords[i, j, :] = Points[node_index]

# Create a dictionary that represents a MATLAB struct with multiple components
matlab_struct = {
    'ConnectivityList': ConnectivityList,
    'Points': Points,
    'triangles': triangles_coords
}

# Save everything into a single .mat file as a struct
savemat('mesh_data_struct_0.25_giusta.mat', matlab_struct)